## Molecular Dynamics Simulation of Heteropolymer Chromatin

This notebook implements a coarse-grained molecular dynamics simulation to study the 3D organization of chromatin (DNA-protein complexes) within the nuclear environment. The simulation uses OpenMM (chunkchromatin is a wrapper) as the underlying physics engine to model the dynamic behavior of chromatin fibers.

### Core Components

**System Setup:**
- **N = 100 monomers**: Represents a chromatin fiber segment with 100 coarse-grained beads aka monomers
- **Density = 0.33**: Sets the packing density within a cubic simulation box
- **Chains**: Defines two polymer chains (0-50, 50-100) representing chromatin domains, False indicates the chains are not circular
- **Monomer types**: This describes the type of each particle

**Force Field Implementation:**
- **Harmonic bonds**: Maintains polymer connectivity with spring-like forces
- **Angle forces**: Preserves chromatin fiber stiffness and persistence length
- **Spherical confinement**: Models nuclear boundary constraints
- **Polynomial repulsive forces**: Prevents monomer overlap (excluded volume effects)
- **Non-bonded pair potentials**: Implements sequence-specific interactions via interaction matrix
- **Lamina interactions**: Pins C-type monomers and attracts B-type monomers to nuclear periphery

### Simulation Protocol

**Production Phase:**
- 100 blocks of 100,000 timesteps each (10 fs timestep)
- Uses variable Langevin integrator at 300K
- HDF5 trajectory output for analysis

### Physical Parameters

- **Temperature**: 300K (physiological conditions)
- **Damping**: γ = 0.05 ps⁻¹ (Langevin thermostat)
- **Interaction matrix**: 3×3 matrix defining monomer-type affinities
- **Box size**: Calculated from monomer count and target density

This simulation framework enables investigation of chromatin organization principles, compartmentalization dynamics, and the role of sequence-specific interactions in 3D genome architecture.


In [ ]:
import argparse
import numpy as np
import os
from chunkchromatin.simulation import Simulation
from chunkchromatin.chromosome import Chromosome
from chunkchromatin.lamina import Lamina
from chunkchromatin.hdf5_format import HDF5Reporter
from chunkchromatin.simulation import EKExceedsError
import openmm as mm
from polykit.polykit.generators.initial_conformations import create_random_walk

import json

In [ ]:

N = 100
density = 0.33
chains = [(0,50,False), (50,100,False)]

#refactor - make this random initialization with same seed
monomer_types = np.load(args.monomer_types)

interaction_matrix = np.array([
    [0.05, 0.05, 0.08],
    [0.05, 0.13, 0.17],
    [0.08, 0.17, 0.22]
])

box_length = (N/density) ** (1/3.)
monomer_positions = create_random_walk(step_size=1, N=N)

out_dir = 'test_output'
os.makedirs(out_dir, exist_ok=True)

reporter = HDF5Reporter(folder=out_dir, max_data_length=500, overwrite=True)

#make a different simulation object for equilibration
sim = Simulation(
    integrator_type="variableLangevin",
    temperature=300.0,  # in Kelvin
    gamma=0.05,         # in 1/ps
    timestep=5,    # in fs
    platform='CPU',
    N=N,
    reporter=reporter
)

chromosome = Chromosome(N, chains, sim)
lamina = Lamina(N, chains,sim)

sim.set_positions(monomer_positions)

harmonic_bond_force = chromosome.add_harmonic_bond()
angle_force = chromosome.add_angle_force()
nonbonded_pair_potential_force = chromosome.add_nonbonded_pair_potential(sim,interaction_matrix,monomer_types)
spherical_confinement_force = lamina.add_spherical_confinement(sim)


sim.add_force(harmonic_bond_force)
sim.add_force(angle_force)
sim.add_force(nonbonded_pair_potential_force)
sim.add_force(spherical_confinement_force)



sim.create_context()
sim.set_velocities()






In [ ]:

for _ in range(100):
    sim.run_simulation_block(1000)
    with open(f"{out_dir}/simulation_stats.txt", "a") as f:
        stats = str(sim.print_stats())
        f.write(stats + "\n")
    reporter.dump_data()